# Chains

In [8]:
from langchain_openai import ChatOpenAI
from IPython.display import display, HTML, Markdown

# Model configuration
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
)

##  Simple single chain using LCEL

LCEL (LangChain Expression Language)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser # Extract the string response only

template = """Your job is to come up with a classic dish from the area that the users suggests.
{location}
 YOUR RESPONSE:
"""

prompt = PromptTemplate.from_template(template) # Create a prompt template

location_chain_lcel = prompt | openai_llm | StrOutputParser()

result = location_chain_lcel.invoke({"location": "China"}) # Invoke the chain with 'China' as the location
print(result)

A classic dish from China is Peking Duck. This renowned dish features crispy skin and tender meat, typically served with thin pancakes, hoisin sauce, and scallions. It's a celebrated delicacy that highlights Chinese culinary artistry and is especially popular in Beijing.


## Simple sequential chain using LCEL

An example of `Multi Agent System`, where we pass output of one LLM to another LLm

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define the templates for each step
location_template = """Your job is to come up with a classic dish from the area that the users suggests.
{location}

YOUR RESPONSE:
"""

dish_template = """Given a meal {meal}, give a short and simple recipe on how to make that dish at home.

YOUR RESPONSE:
"""

time_template = """Given the recipe {recipe}, estimate how much time I need to cook it.

YOUR RESPONSE:
"""

# Create Location chain - This chain takes a location and returns a classic dish from that region
location_chain_lcel = (
    PromptTemplate.from_template(location_template) 
    | openai_llm
    | StrOutputParser()
)

# Create the dish - This chain takes a meal name and returns a recipe
dish_chain_lcel = (
    PromptTemplate.from_template(dish_template)
    | openai_llm
    | StrOutputParser()
)

# Create the time estimation chain - This chain takes a recipe and returns an estimated cooking time
time_chain_lcel = (
    PromptTemplate.from_template(time_template)
    | openai_llm
    | StrOutputParser()
)

# Combine all chains into a single workflow using RunnablePassthrough.assign
# RunnablePassthrough.assign adds new keys to the input dictionary without removing existing ones
overall_chain_lcel = (
    # Step 1: Generate a meal based on location and add it to the input dictionary
    RunnablePassthrough.assign(meal=lambda x: location_chain_lcel.invoke({"location": x["location"]}))
    # Step 2: Generate a recipe based on the meal and add it to the input dictionary
    | RunnablePassthrough.assign(recipe=lambda x: dish_chain_lcel.invoke({"meal": x["meal"]}))
    # Step 3: Estimate cooking time based on the recipe and add it to the input dictionary
    | RunnablePassthrough.assign(time=lambda x: time_chain_lcel.invoke({"recipe": x["recipe"]}))
)
# Run the chain
result = overall_chain_lcel.invoke({"location": "China"})
print(result)

{'location': 'China', 'meal': 'A classic dish from China is Peking Duck. This iconic dish features crispy roasted duck served with thin pancakes, hoisin sauce, and sliced scallions. It originates from Beijing and is renowned for its crispy skin and flavorful meat, making it a must-try traditional Chinese delicacy.', 'recipe': "Here's a simple way to make Peking Duck at home:\n\n**Ingredients:**\n- 1 whole duck (about 4-5 pounds)\n- 2 tablespoons maltose or honey\n- 2 tablespoons soy sauce\n- 1 teaspoon five-spice powder\n- Thin pancakes or crepes\n- Hoisin sauce\n- Sliced scallions\n- Cucumber or other vegetables (optional)\n\n**Instructions:**\n1. **Prepare the duck:** Clean the duck and dry it thoroughly. Mix maltose or honey with soy sauce and five-spice powder. Rub this mixture all over the duck, especially on the skin.\n2. **Air dry:** Hang the duck in a cool, airy place or refrigerate uncovered for several hours or overnight to dry the skin.\n3. **Roast:** Preheat your oven to 37

In [17]:
print(result['location'], end='\n-----------------------\n')
print(result['meal'], end='\n-----------------------\n')
print(result['recipe'], end='\n-----------------------\n')
print(result['time'], end='\n-----------------------\n')

China
-----------------------
A classic dish from China is Peking Duck. This iconic dish features crispy roasted duck served with thin pancakes, hoisin sauce, and sliced scallions. It originates from Beijing and is renowned for its crispy skin and flavorful meat, making it a must-try traditional Chinese delicacy.
-----------------------
Here's a simple way to make Peking Duck at home:

**Ingredients:**
- 1 whole duck (about 4-5 pounds)
- 2 tablespoons maltose or honey
- 2 tablespoons soy sauce
- 1 teaspoon five-spice powder
- Thin pancakes or crepes
- Hoisin sauce
- Sliced scallions
- Cucumber or other vegetables (optional)

**Instructions:**
1. **Prepare the duck:** Clean the duck and dry it thoroughly. Mix maltose or honey with soy sauce and five-spice powder. Rub this mixture all over the duck, especially on the skin.
2. **Air dry:** Hang the duck in a cool, airy place or refrigerate uncovered for several hours or overnight to dry the skin.
3. **Roast:** Preheat your oven to 375°F (